### Lecture Notes: More Examples on Hypothesis Testing & Assessing Models

**Helpful Resource:**
- [Python Reference](http://data8.org/sp22/python-reference.html)

**Recommended Readings:**
- [Assessing a Model](https://www.inferentialthinking.com/chapters/11/1/Assessing_a_Model.html)
- [Decisions and P-value](https://www.inferentialthinking.com/chapters/11/3/Decisions_and_Uncertainty.html)
- [A/B Testing](https://www.inferentialthinking.com/chapters/12/1/AB_Testing.html)
- [Testing two proportions](https://www.inferentialthinking.com/chapters/12/2/Causality.html)

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
students = Table().read_table('student_data.csv')


### Hypothesis Testing Based on Simulations

1. State the null and alternative hypothesis
2. Pick a test statistic
3. Simulate the sampling distribution of the test statistic, under the null hypothesis (model)
4. Draw conclusion based on the results of the random sampling distribution simulation and empirical (real world data) distributions.

### Test Statistic

Test statistic is typically used to make decision on null vs alternative hypothese.

We have learned difference in means, absolute difference in means and total variation distance (TVD).

Difference in means: for 2 numerical groups, compare whether one group performs better or worse that the other in alternative hypothesis.

Absolution difference in means: for 2 numerical groups, compare whether the groups are difference in alternative hypothesis.

Total variation distance (TVD): for multiple categories with different proportions (non numerical data), compare whether the distributions are different.


### Useful `datascience` library functions

`sample_proportions()` returns an array.

`samples()` returns a table.

We may start with simulating one value of the test statistic and then use a `for`-loop to simulate multiple values of the test statistic.

### Use Total Variation Distance (TVD) as Test Statistic and Calculate the Real World Observed TVD Value

1. compute the differences of the control group and the treatment group
2. take the the absolute value of each differences
3. add up all the differences and divide the total by 2

### A/B Testing

In A/B testing, we ask whether the two samples are likely from the same population.

Under the null model, the sampling distribution of the test statistic was generated using a technique called "label shuffling" or "Permutation Shuffling" (random assignment)

### A/B Testing Example 1: Fitness App User Interface and Sleep Quality (Numerical Data)

**Null hypothesis**: Gamified UI has no effect on deep sleep duration.

**Alternative hypothesis**: Gamified UI increases deep sleep duration.

**Test statistic**: the difference between the two sample means (average of deep sleep mins) to evaluate whether the gamified UI will increase in deep sleep duration.

Performing A/B Testing

In [ ]:
sleep_app = Table().read_table('sleep_study_data.csv')
sleep_app.show(3)

In [ ]:
# create a table to contain a label and a variable for A/B testing
#   the labels will be shufflied and randomly associated to the variable column 
ui_and_min = sleep_app.select('UI Mode', 'Mins Deep Sleep')
ui_and_min.show(3)

In [ ]:
ui_and_min.group('UI Mode')

In [ ]:
ui_and_min.hist('Mins Deep Sleep', group='UI Mode')

In [ ]:
# check the average mins 
means_table = ui_and_min.group("UI Mode", np.average)
means_table

In [ ]:
# compute the average based on the real data
means = means_table.column(1)
observed_difference = means.item(0) - means.item(1)
observed_difference

In [ ]:
# to help simulating the sampling distribution of the test statistic,
#   we create a function to compute an average based on 
#   the table, label and variable in the parameter list
def difference_of_means(table, label, variable):
    subtable = table.select(label, variable)
    means_table = subtable.group(label, np.average)
    means = means_table.column(1)
    return means.item(0) - means.item(1)

difference_of_means(sleep_app, "UI Mode", "Mins Deep Sleep")

In [ ]:
# to do A/B testing, we shuffle the labels and randomly re-associated them to variable column
# random assignment
shuffled_labels = ui_and_min.sample(with_replacement = False).column(0)
original_and_shuffled = ui_and_min.with_columns("Shuffled Label", shuffled_labels)
original_and_shuffled

In [ ]:
# check the average after label shuffling
shuffled_only = original_and_shuffled.select("Mins Deep Sleep", "Shuffled Label")
shuffled_group_mean = shuffled_only.group('Shuffled Label', np.average)
shuffled_group_mean

In [ ]:
# call the function to compute the average for the shufflied label
#   this is the average based on the shufflied data
difference_of_means(original_and_shuffled, "Shuffled Label", "Mins Deep Sleep")

In [ ]:
# call the function to compute the average for the original label
#   this is the average based on the real data
difference_of_means(original_and_shuffled, "UI Mode", "Mins Deep Sleep")

In [ ]:
# function to simulate one the sampling distribution of the test statistic
def one_simulated_difference_of_means(table, label, variable):
    # sample the entire table and create shufflied labels
    shuffled_labels = table.sample(with_replacement=False).column(label)
    shuffled_table = table.select(variable).with_column(
        "Shuffled Label", shuffled_labels)
    # call a function to compute a simulated test statistic 
    return difference_of_means(shuffled_table, "Shuffled Label", variable)

In [ ]:
one_simulated_difference_of_means(ui_and_min, "UI Mode", "Mins Deep Sleep")

In [ ]:
# repeat the simulation many times
simulated_differences = make_array()

repeats = 5000
for i in np.arange(repeats):
    new_difference = one_simulated_difference_of_means(ui_and_min, "UI Mode", "Mins Deep Sleep")
    simulated_differences = np.append(simulated_differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', simulated_differences).hist()
print('Observed Difference:', observed_difference)
plots.title('A/B Test: Gamified vs Static UI Sleep Effects');
plots.axvline(observed_difference, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.legend()

In [ ]:
# p-value >= 0.05 support null hypothesis that is the assumption of random sampling
# p-value < 0.05, the data provides strong evidence against the null hypothesis
p_value = np.count_nonzero(simulated_differences >= observed_difference) / repeats
print(f"Calculated Empirical p-value: {p_value:.4f}")

p-value is less than 0.05, we reject the null hypothesis and the gamified UI may increases deep sleep duration.

In [ ]:
students = Table().read_table('student_data.csv')
students.show(3)

### A/B Testing Example 2: Social Media and Average Age

Based on the student data in `student_data.csv`, suppose we want to test the hypothesis that students who do not use social media tend to be older in terms of average age.

**Null hypothesis**: people who don't use social networks have the same mean of age as the rest of the students.

**Alternative hypothesis**: people who don't use social networks have a higher mean of age as the rest of the students.

**Test statistic**: the difference between the two sample means (the average age)

Performing A/B Testing

In [ ]:
# create a table to contain a label and a variable for A/B testing
#   the labels will be shufflied and randomly associated to the variable column 
social_and_age = students.select("SOCIAL", "AGE")
social_and_age

In [ ]:
# use tbl.group() function to count the A and B categories
social_and_age.group("SOCIAL")

In [ ]:
# visualize the distribution
social_and_age.hist("AGE", group="SOCIAL")

In [ ]:
# check the average age 
means_table = social_and_age.group("SOCIAL", np.average)
means_table

In [ ]:
# compute the average based on the real data
means = means_table.column(1)
observed_difference = means.item(0) - means.item(1)
observed_difference

In [ ]:
# to help simulating the sampling distribution of the test statistic,
#   we create a function to compute an average based on 
#   the table, label and variable in the parameter list
def difference_of_means(table, label, variable):
    subtable = table.select(label, variable)
    means_table = subtable.group(label, np.average)
    means = means_table.column(1)
    return means.item(0) - means.item(1)

observed_diff = difference_of_means(students, "SOCIAL", "AGE")
observed_diff

In [ ]:
# to do A/B testing, we shuffle the labels and randomly re-associated them to variable column
# random assignment
shuffled_labels = social_and_age.sample(with_replacement = False).column(0)
original_and_shuffled = social_and_age.with_columns("Shuffled Label", shuffled_labels)
original_and_shuffled

In [ ]:
# check the average after label shuffling
shuffled_only = original_and_shuffled.select("AGE", "Shuffled Label")
shuffled_group_mean = shuffled_only.group('Shuffled Label', np.average)
shuffled_group_mean

In [ ]:
# call the function to compute the average for the shufflied label
#   this is the average based on the shufflied data
difference_of_means(original_and_shuffled, "Shuffled Label", "AGE")

In [ ]:
# call the function to compute the average for the original label
#   this is the average based on the real data
difference_of_means(original_and_shuffled, "SOCIAL", "AGE")

In [ ]:
# function to simulate one the sampling distribution of the test statistic
def one_simulated_difference_of_means(table, label, variable):
    # sample the entire table and create shufflied labels
    shuffled_labels = table.sample(with_replacement=False).column(label)
    shuffled_table = table.select(variable).with_column(
        "Shuffled Label", shuffled_labels)
    # call a function to compute a simulated test statistic 
    return difference_of_means(shuffled_table, "Shuffled Label", variable)

In [ ]:
one_simulated_difference_of_means(social_and_age, "SOCIAL", "AGE")

In [ ]:
# repeat the simulation many times
simulated_differences = make_array()

repeats = 5000
for i in np.arange(repeats):
    new_difference = one_simulated_difference_of_means(social_and_age, "SOCIAL", "AGE")
    simulated_differences = np.append(simulated_differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', simulated_differences).hist()
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');
#plots.scatter(observed_difference, 0.005, color='red', s=100)
plots.axvline(observed_difference, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.legend()

In [ ]:
# p-value >= 0.05 support null hypothesis that is the assumption of random sampling
# p-value < 0.05, the data provides strong evidence against the null hypothesis
p_value = np.count_nonzero(simulated_differences >= observed_difference) / repeats
p_value

So the data is consistent with the null. There is no evidence to show people who don't use social network are older.

### A/B Testing Example 3: Lithium Prevent Cocaine Users from Relapsing (Categorical Data)

Assume the following data records an experiment on whether Lithium was effective in preventing cocaine users from relapsing (returning to cocaine use). In the column labeled "Result", 1 represents no relapse (success), and 0 presents relapse (failure).

**Null hypothesis**: (control group) Lithium has no effect on the success rate in preventing cocaine users from relapsing.

**Alternative hypothesis**: (treatment group) Lithium does have an effect on the success rate in preventing cocaine users from relapsing.

**Test statistic**: the absolute value of the difference in the success rate between the two groups (small absolute values support the null hypothesis, otherwise support the alternative hypothesis)

Performing A/B Testing (label shuffling) - random assignment

In [ ]:
coke = Table().read_table('cocaine_lithium.csv')
coke.show(3)

In [ ]:
# the table doesn't really have numerical values
#   the 1s and 0s represent boolean values
# so histogram doesn't make much sense
# remember histogram is used to visual the distribution of continuous numerical data
coke.hist(group="Group")

In [ ]:
# let's summerize the data using a pivot table
coke.pivot('Result', 'Group')

In [ ]:
# average success rates
coke_avg = coke.group('Group', np.average)
coke_avg

In [ ]:
# how far/close of the success rate between the control and treatment groups
observed_distance = abs(coke_avg.column('Result average').item(1) - coke_avg.column('Result average').item(0))
observed_distance

In [ ]:
# create a function to computer the distance between the control and treatment groups
def distance(table, group_label):
    proportions = table.group(group_label, np.average).column(1)
    #print(proportions)
    return abs(proportions.item(1) - proportions.item(0))

distance(coke, 'Group')

In [ ]:
# to do A/B testing, we shuffle the labels and randomly re-associated them to variable column
# random assignment
shuffled_labels = coke.sample(with_replacement = False).column(0)
original_and_shuffled = coke.with_columns("Shuffled Label", shuffled_labels)
original_and_shuffled.show(5)

In [ ]:
shuffled_only = original_and_shuffled.select("Result", "Shuffled Label")

# call the function to compute the distance (absolute value) for the shufflied label
#   this is the distance (absolute value) based on the shufflied data
distance(shuffled_only, "Shuffled Label")

In [ ]:
distance(coke, 'Group')

In [ ]:
# function to simulate one the sampling distribution of the test statistic
def one_simulated_distance(table, label, variable):
    # sample the entire table and create shufflied labels
    shuffled_labels = table.sample(with_replacement=False).column(label)
    shuffled_table = table.select(variable).with_column(
        "Shuffled Label", shuffled_labels)
    # call a function to compute a simulated test statistic 
    return distance(shuffled_table, "Shuffled Label")

In [ ]:
one_simulated_distance(coke, "Group", "Result")

In [ ]:
# repeat the simulation many times
simulated_distances = make_array()

repeats = 10000
for i in np.arange(repeats):
    new_distance = one_simulated_distance(coke, "Group", "Result")
    simulated_distances = np.append(simulated_distances, new_distance)

Table().with_column('abs diff in success rates', simulated_distances).hist()

print('Observed Difference:', observed_distance)
plots.title('Prediction Under the Null Hypothesis');
#plots.scatter(observed_difference, 0.005, color='red', s=100)
plots.axvline(observed_distance, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.legend()

#### Would you recommend this treament to anyone?

#### What do you think the p-value limit should be?

#### let's look at the p-value

In [ ]:
# let's look at the p-value
# p-value >= 0.05 support null hypothesis that is the assumption of random sampling
# p-value < 0.05, the data provides strong evidence against the null hypothesis
np.count_nonzero(simulated_distances >= observed_distance) / repeats

### A/B Testing Example 4: Social Media and Assignments submissions

**Null hypothesis**: students use social media submit similar numbers of assignments on average as the students don't use social media

**Alternative hypothesis**: students use social media submit less assignments on average than the students don't use social media

**Test statistic**: the difference between the two sample means (the average number of assignments submitted)


In [ ]:
students.show(3)

In [ ]:
hw = students.column('MATH') + students.column('PAPER')
social_and_hw = Table().with_columns('SOCIAL', students.column('SOCIAL'), 'HOMEWORK', hw)
social_and_hw.show(3)

In [ ]:
social_and_hw.group('SOCIAL')

In [ ]:
social_and_hw.pivot('HOMEWORK', 'SOCIAL')

In [ ]:
social_and_hw.hist('HOMEWORK', bins=np.arange(1, 12, 1))

In [ ]:
social_and_hw.hist("HOMEWORK", group="SOCIAL")

In [ ]:
means_table = social_and_hw.group("SOCIAL", np.average)
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(0) - means.item(1)
observed_difference

In [ ]:
observed_diff = difference_of_means(social_and_hw, "SOCIAL", "HOMEWORK")
observed_diff

In [ ]:
shuffled_labels = social_and_hw.sample(with_replacement = False).column(0)
original_and_shuffled = social_and_hw.with_columns("Shuffled Label", shuffled_labels)
original_and_shuffled

In [ ]:
shuffled_only = original_and_shuffled.select("HOMEWORK", "Shuffled Label")
shuffled_group_mean = shuffled_only.group('Shuffled Label', np.average)
shuffled_group_mean

In [ ]:
difference_of_means(original_and_shuffled, "Shuffled Label", "HOMEWORK")

In [ ]:
difference_of_means(original_and_shuffled, "SOCIAL", "HOMEWORK")

In [ ]:
one_simulated_difference_of_means(social_and_hw, "SOCIAL", "HOMEWORK")

In [ ]:
simulated_differences = make_array()

repeats = 5000
for i in np.arange(repeats):
    new_difference = one_simulated_difference_of_means(social_and_hw, "SOCIAL", "HOMEWORK")
    simulated_differences = np.append(simulated_differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', simulated_differences).hist()
print('Observed Difference:', observed_difference)
plots.axvline(observed_difference, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.legend()

In [ ]:
# p-value >= 0.05 support null hypothesis that is the assumption of random sampling
# p-value < 0.05, the data provides strong evidence against the null hypothesis
p_value = np.count_nonzero(simulated_differences <= observed_difference) / repeats
p_value